# 0.1. Ustawienie ścieżek

In [1]:
import sys
from abc import ABC
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "src"))

from config.config import setup

cfg = setup()

HAR_ROOT = cfg["HAR_ROOT"]
MODELS_DIR = cfg["MODELS_DIR"]
RESULTS_DIR = cfg["RESULTS_DIR"]

# 0.2. Wgranie danych z datasetu

In [2]:
from data.loader import load_har_data

X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names = (
    load_har_data(HAR_ROOT)
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (7352, 561) | X_test: (2947, 561)


# 0.3. Konfiguracja danych treningowych i parametrów eksperymentu (CV, metryka, wyniki)

In [3]:
import numpy as np
from sklearn.model_selection import GroupKFold
from src.tuning.grid_search_cv import run_grid
from src.config.constants import N_JOBS

y_train_arr = np.asarray(y_train)
groups_train_arr = np.asarray(groups_train)

CV = GroupKFold(n_splits=5)
SCORING = "matthews_corrcoef"
ALL_SUMMARIES = []

# 1. Sprawdzenie rozkładu osób w foldach GroupKFold(5):

In [4]:
print("Sprawdzenie rozkładu osób w foldach GroupKFold(5):")
for i, (tr, va) in enumerate(
    CV.split(X_train, y_train_arr, groups=groups_train_arr), 1
):
    train_subj = np.unique(groups_train_arr[tr]).astype(int).tolist()
    val_subj = np.unique(groups_train_arr[va]).astype(int).tolist()
    print(
        f"fold {i}: train_subj={str(train_subj):<65}  |  val_subj={str(val_subj):<20}  ({len(va)} okien)"
    )

Sprawdzenie rozkładu osób w foldach GroupKFold(5):
fold 1: train_subj=[1, 3, 5, 6, 7, 8, 11, 16, 17, 21, 22, 23, 26, 27, 28, 29, 30]     |  val_subj=[14, 15, 19, 25]      (1420 okien)
fold 2: train_subj=[1, 3, 5, 7, 8, 11, 14, 15, 17, 19, 23, 25, 26, 27, 28, 29, 30]    |  val_subj=[6, 16, 21, 22]       (1420 okien)
fold 3: train_subj=[1, 5, 6, 7, 8, 14, 15, 16, 19, 21, 22, 23, 25, 27, 28, 29, 30]    |  val_subj=[3, 11, 17, 26]       (1417 okien)
fold 4: train_subj=[3, 5, 6, 8, 11, 14, 15, 16, 17, 19, 21, 22, 25, 26, 27, 28, 29]   |  val_subj=[1, 7, 23, 30]        (1410 okien)
fold 5: train_subj=[1, 3, 6, 7, 11, 14, 15, 16, 17, 19, 21, 22, 23, 25, 26, 30]       |  val_subj=[5, 8, 27, 28, 29]    (1685 okien)


# 2. DummyClassifier – strojenie hiperparametrów

In [5]:
from models.pipelines import pipe_dummy

param_grid_dummy = {
    "clf__strategy": ["stratified", "most_frequent", "uniform"],
}

grid_dummy, s = run_grid(
    "dummy",
    pipe_dummy,
    param_grid_dummy,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 3 candidates, totalling 15 fits

[dummy]	Best MCC = 0.0034		|		3 kombinacji × 5 foldów = 15 fitów		|		czas = 0.3s
[dummy]	Best_params: {'clf__strategy': 'uniform'}


# 3. LogisticRegression

In [ ]:
from models.pipelines import pipe_logreg

# param_grid_logreg = [
#     {
#         "clf__C":        [0.01, 0.1, 1.0, 10.0],
#         "clf__l1_ratio": [0.0],
#         "clf__solver":   ["lbfgs", "saga", "newton-cg", "sag"],
#     },
#     {
#         "clf__C":        [0.01, 0.1, 1.0, 10.0],
#         "clf__l1_ratio": [1.0],
#         "clf__solver":   ["saga"],
#     },
# ]
param_grid_logreg = [
    {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__solver": ["lbfgs", "saga"],
    },
    {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__l1_ratio": [1.0],
        "clf__solver": ["saga"],
    },
]

grid_logreg, s = run_grid(
    "logreg",
    pipe_logreg,
    param_grid_logreg,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

# 4. LinearSVC

In [ ]:
from models.pipelines import pipe_linear_svc

param_grid_lsvc = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__loss": ["squared_hinge"],
}

grid_lsvc, s = run_grid(
    "linear_svc",
    pipe_linear_svc,
    param_grid_lsvc,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


# 5. SVC (RBF)

In [ ]:
from models.pipelines import pipe_rbf_svc

param_grid_rbf = {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__gamma": ["scale", 0.01, 0.001],
}

grid_rbf, s = run_grid(
    "rbf_svc",
    pipe_rbf_svc,
    param_grid_rbf,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

# 6. RandomForest

In [5]:
from models.pipelines import pipe_random_forest

param_grid_rf = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 20],
    "clf__min_samples_split": [2, 10],
    "clf__max_features": ["sqrt"],
}

grid_rf, s = run_grid(
    "random_forest",
    pipe_random_forest,
    param_grid_rf,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=1,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 8 candidates, totalling 40 fits

[random_forest]	Best MCC = 0.9041		|		8 kombinacji × 5 foldów = 40 fitów		|		czas = 98.4s
[random_forest]	Best_params: {'clf__max_depth': None, 'clf__max_features': 'sqrt', 'clf__min_samples_split': 10, 'clf__n_estimators': 300}


# 7. GaussianNB

In [4]:
from models.pipelines import pipe_gaussian_nb

param_grid_gnb = {
    "clf__var_smoothing": [1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5],
}

grid_gnb, s = run_grid(
    "gaussian_nb",
    pipe_gaussian_nb,
    param_grid_gnb,
    X=X_train,
    y=y_train_arr,
    groups=groups_train_arr,
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    cv=CV,
    scoring=SCORING,
    n_jobs=N_JOBS,
)

ALL_SUMMARIES.append(s)

Fitting 5 folds for each of 6 candidates, totalling 30 fits

[gaussian_nb]	Best MCC = 0.6911		|		6 kombinacji × 5 foldów = 30 fitów		|		czas = 3.6s
[gaussian_nb]	Best_params: {'clf__var_smoothing': 1e-05}
